[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1opf1igX6pKVYlLkD0dJ5XkbLwdW-sbWw)

## Loģistiskā regresija

In [61]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

In [3]:
!wget -q -O data.csv https://raw.githubusercontent.com/IvoDz/lv-text-complexity/refs/heads/main/data.csv

In [18]:
df = pd.read_csv("data.csv")

In [19]:
le = LabelEncoder()
y_enc = le.fit_transform(df["level"])
X = df["text"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

In [21]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000)),
])

param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2), (1,3)],
    'tfidf__analyzer': ['word', 'char_wb'],
    'tfidf__min_df': [1, 3],
    'tfidf__max_df': [0.9, 1.0],
    'clf__C': [0.1, 1.0, 10],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    verbose=2,
    n_jobs=-1,
)

grid.fit(X_train, y_train)

y_pred = grid.predict(X_test)
print("Best params:", grid.best_params_)
print("F1 (macro):", grid.best_score_)
print(classification_report(y_test, y_pred, target_names=le.classes_))

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best params: {'clf__C': 1.0, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs', 'tfidf__analyzer': 'char_wb', 'tfidf__max_df': 1.0, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 3)}
F1 (macro): 0.6478045639189609
              precision    recall  f1-score   support

   sarežģīts       0.69      0.67      0.68        98
      vidējs       0.44      0.46      0.45       109
      viegls       0.67      0.66      0.67       106

    accuracy                           0.59       313
   macro avg       0.60      0.60      0.60       313
weighted avg       0.60      0.59      0.60       313



In [24]:
joblib.dump(grid.best_estimator_, 'logreg_best.pkl')

['logreg_best.pkl']

In [51]:
def run_inference(text: str):
    y_pred = grid.predict([text])
    return le.inverse_transform(y_pred)[0]

In [52]:
run_inference("Vienkāršs teksts")

'viegls'

In [53]:
run_inference("Arī vienkāršs teksts, tikai nedaudz garāks.")

'viegls'

In [56]:
run_inference("Nedaudz sarežģītāks teksts, kas satur dažādas vārdformas.")

'vidējs'

In [58]:
run_inference("""
              Latvijas Universitāte (LU; latīņu: Universitas Latviensis, lībiešu: Lețmō Iļīzskūol)
              ir Latvijas Republikas akadēmiskās un profesionālās augstākās izglītības un zinātnes institūcija")
              """)


'sarežģīts'